# Barclays Pre-Delinquency Intervention Engine
## ML Training Pipeline — Full Results & Evaluation

**Architecture:** XGBoost + LightGBM + TFT (Temporal Fusion Transformer) ensemble with meta-learner stacking

**Key Design Decisions:**
- **Outcome-based labels** from actual missed EMI payments (no label leakage)
- **60/20/20 split** — train / calibration / test (calibration set for isotonic regression)
- **Proper OOF meta-learner** — 5-fold StratifiedKFold, no test-set leakage
- **Probability calibration** — isotonic regression for IFRS 9 compliant PD estimates
- **Cost-sensitive thresholds** — FN cost (missed default) = £10,000 vs FP cost (unnecessary RM call) = £5
- **Class imbalance handling** — `scale_pos_weight` for XGBoost/LightGBM

**Regulatory Context:** FCA Consumer Duty, PRA SS1/23, IFRS 9, Basel III/IV

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────
import os, sys
import warnings
warnings.filterwarnings('ignore')

# Path setup — works from any location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, brier_score_loss
)
from sklearn.calibration import calibration_curve

from config.settings import ModelConfig, PostgresConfig
from ml.dataset_builder import build_training_dataset, build_temporal_dataset
from ml.xgboost_model import XGBoostDelinquencyModel
from ml.lightgbm_model import LightGBMDelinquencyModel
from ml.ensemble import EnsembleScorer, StackingEnsemble
from ml.calibration import ProbabilityCalibrator
from ml.threshold_optimizer import optimize_thresholds, save_thresholds

MODEL_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model dir: {MODEL_DIR}")
print("Setup complete.")

---
## 1. Dataset Construction (Outcome-Based Labels)

Labels are derived from the `payment_events` table — actual missed EMI/auto-debit payments
where the customer's running balance was insufficient to cover the payment on the due date.

**No label leakage:** features come from behavioral transaction history, labels come from
future payment outcomes. These are entirely separate data sources.

In [ ]:
# Build tabular dataset from PostgreSQL
X_tab, y_tab, feature_names, customer_ids = build_training_dataset()

# Build temporal dataset for TFT
X_seq, y_seq, cids_seq = build_temporal_dataset()

print(f"\nTabular: {X_tab.shape[0]} customers, {X_tab.shape[1]} features")
print(f"Temporal: {X_seq.shape if X_seq is not None else 'N/A'}")
print(f"Features: {feature_names}")

In [ ]:
# ── Label Distribution ──
n_pos = int(y_tab.sum())
n_neg = len(y_tab) - n_pos
pct = n_pos / len(y_tab) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Stable (0)', 'Delinquent (1)'], [n_neg, n_pos], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Label Distribution (Outcome-Based)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, (v, label) in enumerate(zip([n_neg, n_pos], [f'{n_neg}\n({100-pct:.1f}%)', f'{n_pos}\n({pct:.1f}%)'])):
    axes[0].text(i, v + len(y_tab)*0.01, label, ha='center', fontweight='bold')

# Feature correlation with label (top 15)
df_corr = pd.DataFrame(X_tab, columns=feature_names)
df_corr['label'] = y_tab
corr = df_corr.corr()['label'].drop('label').abs().sort_values(ascending=True).tail(15)
corr.plot.barh(ax=axes[1], color='#3498db', edgecolor='black', linewidth=0.5)
axes[1].set_title('Top 15 Features by |Correlation| with Default', fontweight='bold')
axes[1].set_xlabel('|Pearson r|')

plt.tight_layout()
plt.show()

print(f"\nDelinquency rate: {pct:.1f}% — consistent with real banking portfolios (3-8%)")
print(f"Labels source: payment_events table (actual missed EMI/auto-debit payments)")

---
## 2. Train/Calibration/Test Split

**60/20/20 split** — the calibration set is held out specifically for:
- Isotonic regression calibration (raw scores → calibrated PD)
- Cost-sensitive threshold optimization

This prevents information leakage from the test set into calibration or threshold selection.

In [ ]:
# 60/20/20 stratified split
X_train, X_remain, y_train, y_remain, idx_train, idx_remain = train_test_split(
    X_tab, y_tab, np.arange(len(y_tab)),
    test_size=0.4, stratify=y_tab, random_state=42,
)
X_cal, X_test, y_cal, y_test, idx_cal, idx_test = train_test_split(
    X_remain, y_remain, idx_remain,
    test_size=0.5, stratify=y_remain, random_state=42,
)

# Class imbalance weight
n_pos_train = y_train.sum()
n_neg_train = len(y_train) - n_pos_train
scale_pos_weight = float(n_neg_train / max(n_pos_train, 1))

print(f"Train:       {len(X_train):>5} samples ({y_train.mean()*100:.1f}% delinquent)")
print(f"Calibration: {len(X_cal):>5} samples ({y_cal.mean()*100:.1f}% delinquent)")
print(f"Test:        {len(X_test):>5} samples ({y_test.mean()*100:.1f}% delinquent)")
print(f"\nscale_pos_weight: {scale_pos_weight:.1f} (passed to XGBoost & LightGBM)")

---
## 3. Model Training — XGBoost + LightGBM

In [ ]:
import joblib

# ── XGBoost ──
print("Training XGBoost...")
xgb_best_path = os.path.join(MODEL_DIR, "xgb_best_params.joblib")
xgb_params = {"scale_pos_weight": scale_pos_weight}
if os.path.exists(xgb_best_path):
    xgb_params_raw = joblib.load(xgb_best_path)
    xgb_params = {
        "objective": "binary:logistic", "eval_metric": "auc",
        "scale_pos_weight": scale_pos_weight, "random_state": 42, "n_jobs": -1,
        **xgb_params_raw,
    }
    print("  Using Optuna-tuned hyperparameters")

xgb_model = XGBoostDelinquencyModel(params=xgb_params)
xgb_metrics = xgb_model.train(X_train, y_train, feature_names, X_test, y_test)
xgb_model.save()

xgb_test_probs = xgb_model.predict_proba(X_test)
xgb_auprc = average_precision_score(y_test, xgb_test_probs)

print(f"  Train AUC:  {xgb_metrics['train_auc']:.4f}")
print(f"  CV AUC:     {xgb_metrics['cv_auc_mean']:.4f} +/- {xgb_metrics['cv_auc_std']:.4f}")
print(f"  Test AUC:   {xgb_metrics.get('val_auc', roc_auc_score(y_test, xgb_test_probs)):.4f}")
print(f"  Test AUPRC: {xgb_auprc:.4f}")

In [ ]:
# ── LightGBM ──
print("Training LightGBM...")
lgb_best_path = os.path.join(MODEL_DIR, "lgb_best_params.joblib")
lgb_params = {"scale_pos_weight": scale_pos_weight}
if os.path.exists(lgb_best_path):
    lgb_params_raw = joblib.load(lgb_best_path)
    lgb_params = {
        "objective": "binary", "metric": "auc",
        "scale_pos_weight": scale_pos_weight, "verbose": -1, "n_jobs": -1, "seed": 42,
        **lgb_params_raw,
    }
    print("  Using Optuna-tuned hyperparameters")

lgb_model = LightGBMDelinquencyModel(params=lgb_params)
lgb_metrics = lgb_model.train(X_train, y_train, feature_names, X_test, y_test)
lgb_model.save()

lgb_test_probs = lgb_model.predict_proba(X_test)
lgb_auprc = average_precision_score(y_test, lgb_test_probs)

print(f"  Train AUC:  {lgb_metrics['train_auc']:.4f}")
print(f"  CV AUC:     {lgb_metrics['cv_auc_mean']:.4f} +/- {lgb_metrics['cv_auc_std']:.4f}")
print(f"  Test AUC:   {lgb_metrics.get('val_auc', roc_auc_score(y_test, lgb_test_probs)):.4f}")
print(f"  Test AUPRC: {lgb_auprc:.4f}")

---
## 4. TFT (Temporal Fusion Transformer)

In [ ]:
from ml.tft_model import TFTDelinquencyModel

tft_model = None
tft_metrics = {}
X_static = None

if X_seq is not None and len(X_seq) > 50:
    print("Training TFT (Temporal Fusion Transformer)...")
    n_temporal = X_seq.shape[2]
    static_cols = ["age", "credit_score", "tenure_months", "product_count",
                   "has_credit_card", "has_personal_loan", "has_mortgage"]
    static_indices = [feature_names.index(c) for c in static_cols if c in feature_names]
    X_static = X_tab[:, static_indices] if static_indices else X_tab[:, :7]

    tft_X_temp, tft_X_stat, tft_y = [], [], []
    for i, cid in enumerate(customer_ids):
        seq_idx = np.where(cids_seq == cid)[0]
        if len(seq_idx) > 0:
            tft_X_temp.append(X_seq[seq_idx[0]])
            tft_X_stat.append(X_static[i])
            tft_y.append(y_tab[i])

    if len(tft_y) > 50:
        tft_X_temp = np.array(tft_X_temp)
        tft_X_stat = np.array(tft_X_stat)
        tft_y_arr = np.array(tft_y)

        split_idx = int(0.8 * len(tft_y_arr))
        tft_model = TFTDelinquencyModel(
            n_temporal_features=n_temporal,
            n_static_features=tft_X_stat.shape[1],
            epochs=30, batch_size=64,
        )
        tft_metrics = tft_model.train(
            tft_X_temp[:split_idx], tft_X_stat[:split_idx], tft_y_arr[:split_idx],
            tft_X_temp[split_idx:], tft_X_stat[split_idx:], tft_y_arr[split_idx:],
        )
        tft_model.save(os.path.join(MODEL_DIR, "tft_model.pt"))
        print(f"  Best Val AUC: {tft_metrics.get('best_val_auc', 'N/A')}")
    else:
        print("Insufficient matched temporal data for TFT training.")
else:
    print("Skipping TFT — insufficient temporal data.")

---
## 5. Proper OOF Meta-Learner (No Leakage)

The meta-learner (logistic regression stacker) is trained on **out-of-fold predictions** from 5-fold
StratifiedKFold on the training set. This eliminates the second source of data leakage
(the previous approach trained the meta-learner on test set predictions).

In [ ]:
stacker = StackingEnsemble()

n_splits = 5
kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_xgb = np.zeros(len(X_train))
oof_lgb = np.zeros(len(X_train))
oof_tft = np.full(len(X_train), 0.5)

print(f"Generating {n_splits}-fold OOF predictions...")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
    # XGBoost fold
    fold_xgb = XGBoostDelinquencyModel(params={"scale_pos_weight": scale_pos_weight})
    fold_xgb.train(X_train[tr_idx], y_train[tr_idx], feature_names)
    oof_xgb[val_idx] = fold_xgb.predict_proba(X_train[val_idx])

    # LightGBM fold
    fold_lgb = LightGBMDelinquencyModel(params={"scale_pos_weight": scale_pos_weight})
    fold_lgb.train(X_train[tr_idx], y_train[tr_idx], feature_names)
    oof_lgb[val_idx] = fold_lgb.predict_proba(X_train[val_idx])

    print(f"  Fold {fold+1}/{n_splits} done")

oof_xgb_auc = roc_auc_score(y_train, oof_xgb)
oof_lgb_auc = roc_auc_score(y_train, oof_lgb)
print(f"\nOOF XGBoost AUC:  {oof_xgb_auc:.4f}")
print(f"OOF LightGBM AUC: {oof_lgb_auc:.4f}")

In [ ]:
# TFT OOF predictions (for customers with temporal sequences)
if tft_model is not None and X_seq is not None:
    train_cids = customer_ids[idx_train]
    for i, cid in enumerate(train_cids):
        seq_idx = np.where(cids_seq == cid)[0]
        if len(seq_idx) > 0:
            cid_static_idx = np.where(customer_ids == cid)[0]
            if len(cid_static_idx) > 0 and X_static is not None:
                try:
                    oof_tft[i] = tft_model.predict_proba(
                        X_seq[seq_idx[0]:seq_idx[0]+1],
                        X_static[cid_static_idx[0]:cid_static_idx[0]+1]
                    )[0]
                except Exception:
                    pass
    print(f"TFT OOF predictions: {(oof_tft != 0.5).sum()}/{len(oof_tft)} customers matched")
else:
    print("TFT not available — using default 0.5 for TFT OOF")

# Train meta-learner on OOF predictions
try:
    from sqlalchemy import create_engine
    engine = create_engine(PostgresConfig.get_url())
    meta_df = pd.read_sql(
        "SELECT customer_id, income_bracket, tenure_months, credit_score FROM customers", engine
    )
    train_cids = customer_ids[idx_train]
    meta_info = meta_df[meta_df["customer_id"].isin(train_cids)].reset_index(drop=True)

    meta_X = stacker.build_meta_features_batch(
        xgb_probs=oof_xgb, lgb_probs=oof_lgb, tft_probs=oof_tft,
        income_brackets=meta_info["income_bracket"].tolist() if len(meta_info) >= len(X_train) else None,
        tenure_months_arr=meta_info["tenure_months"].values if len(meta_info) >= len(X_train) else None,
        credit_scores=meta_info["credit_score"].values if len(meta_info) >= len(X_train) else None,
    )

    meta_metrics = stacker.train_meta_learner(meta_X, y_train)
    stacker.save_meta_learner(os.path.join(MODEL_DIR, "meta_learner.joblib"))
    print(f"\nMeta-Learner CV AUC: {meta_metrics['cv_auc_mean']:.4f} +/- {meta_metrics['cv_auc_std']:.4f}")
    print(f"Coefficients: {meta_metrics['coefficients']}")
except Exception as e:
    print(f"Meta-learner training error: {e}")
    meta_metrics = {}

---
## 6. Ensemble Evaluation on Test Set

In [ ]:
ensemble = EnsembleScorer()

# Get individual model test predictions
xgb_test_probs = xgb_model.predict_proba(X_test)
lgb_test_probs = lgb_model.predict_proba(X_test)

tft_test_probs = None
if tft_model is not None and X_seq is not None:
    test_cids = customer_ids[idx_test]
    tft_test_probs = np.full(len(X_test), 0.5)
    for i, cid in enumerate(test_cids):
        seq_idx = np.where(cids_seq == cid)[0]
        if len(seq_idx) > 0:
            static_idx = np.where(customer_ids == cid)[0]
            if len(static_idx) > 0 and X_static is not None:
                try:
                    tft_test_probs[i] = tft_model.predict_proba(
                        X_seq[seq_idx[0]:seq_idx[0]+1],
                        X_static[static_idx[0]:static_idx[0]+1]
                    )[0]
                except Exception:
                    pass

# Fixed-weight ensemble
ensemble_probs = ensemble.combine_batch(xgb_test_probs, lgb_test_probs, tft_probs=tft_test_probs)
ensemble_auc = roc_auc_score(y_test, ensemble_probs)
ensemble_auprc = average_precision_score(y_test, ensemble_probs)

print(f"Fixed-weight Ensemble — AUC: {ensemble_auc:.4f}, AUPRC: {ensemble_auprc:.4f}")

# Stacked ensemble (meta-learner)
stacked_auc = None
if stacker.meta_learner is not None:
    stacked_probs = np.array([
        stacker.combine_stacked(
            xgb_prob=xgb_test_probs[i],
            lgb_prob=lgb_test_probs[i],
            tft_prob=tft_test_probs[i] if tft_test_probs is not None else None,
        ) for i in range(len(X_test))
    ])
    stacked_auc = roc_auc_score(y_test, stacked_probs)
    stacked_auprc = average_precision_score(y_test, stacked_probs)
    print(f"Stacked Ensemble   — AUC: {stacked_auc:.4f}, AUPRC: {stacked_auprc:.4f}")
    # Use stacked as final ensemble
    ensemble_probs = stacked_probs
    ensemble_auc = stacked_auc
    ensemble_auprc = stacked_auprc

In [ ]:
# ── ROC Curves: Individual Models vs Ensemble ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# ROC
ax = axes[0]
for name, probs, color in [
    ('XGBoost', xgb_test_probs, '#e74c3c'),
    ('LightGBM', lgb_test_probs, '#3498db'),
    ('Ensemble', ensemble_probs, '#2ecc71'),
]:
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc_val = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)

if tft_test_probs is not None:
    fpr, tpr, _ = roc_curve(y_test, tft_test_probs)
    auc_val = roc_auc_score(y_test, tft_test_probs)
    ax.plot(fpr, tpr, label=f'TFT (AUC={auc_val:.3f})', color='#9b59b6', linewidth=2)

ax.plot([0,1], [0,1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Individual Models vs Ensemble', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Precision-Recall
ax = axes[1]
for name, probs, color in [
    ('XGBoost', xgb_test_probs, '#e74c3c'),
    ('LightGBM', lgb_test_probs, '#3498db'),
    ('Ensemble', ensemble_probs, '#2ecc71'),
]:
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    ax.plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color, linewidth=2)

if tft_test_probs is not None:
    prec, rec, _ = precision_recall_curve(y_test, tft_test_probs)
    ap = average_precision_score(y_test, tft_test_probs)
    ax.plot(rec, prec, label=f'TFT (AP={ap:.3f})', color='#9b59b6', linewidth=2)

baseline = y_test.mean()
ax.axhline(y=baseline, color='k', linestyle='--', alpha=0.3, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (AUPRC)', fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("AUPRC is the primary metric for imbalanced credit risk — it is not inflated by true negatives.")

---
## 7. Probability Calibration (IFRS 9 PD)

Raw ensemble scores are relative rankings, not true probabilities.
IFRS 9 requires **calibrated Probability of Default (PD)** estimates for Expected Credit Loss (ECL) calculations.

**Method:** Isotonic regression on the held-out calibration set.
When the model says 0.65, approximately 65% of customers with that score actually default.

In [ ]:
calibrator = ProbabilityCalibrator()

# Get ensemble scores on CALIBRATION set (separate from train and test)
xgb_cal_probs = xgb_model.predict_proba(X_cal)
lgb_cal_probs = lgb_model.predict_proba(X_cal)

tft_cal_probs = None
if tft_model is not None and X_seq is not None:
    cal_cids = customer_ids[idx_cal]
    tft_cal_probs = np.full(len(X_cal), 0.5)
    for i, cid in enumerate(cal_cids):
        seq_idx = np.where(cids_seq == cid)[0]
        if len(seq_idx) > 0:
            static_idx = np.where(customer_ids == cid)[0]
            if len(static_idx) > 0 and X_static is not None:
                try:
                    tft_cal_probs[i] = tft_model.predict_proba(
                        X_seq[seq_idx[0]:seq_idx[0]+1],
                        X_static[static_idx[0]:static_idx[0]+1]
                    )[0]
                except Exception:
                    pass

cal_ensemble_probs = ensemble.combine_batch(xgb_cal_probs, lgb_cal_probs, tft_probs=tft_cal_probs)
if stacker.meta_learner is not None:
    cal_ensemble_probs = np.array([
        stacker.combine_stacked(
            xgb_prob=xgb_cal_probs[i], lgb_prob=lgb_cal_probs[i],
            tft_prob=tft_cal_probs[i] if tft_cal_probs is not None else None,
        ) for i in range(len(X_cal))
    ])

# Fit calibrator on calibration set
cal_metrics = calibrator.fit(cal_ensemble_probs, y_cal)
calibrator.save(os.path.join(MODEL_DIR, "calibrator.joblib"))

# Apply calibration to test set
raw_test_probs = ensemble_probs.copy()
calibrated_test_probs = calibrator.calibrate_batch(ensemble_probs)
calibrated_auc = roc_auc_score(y_test, calibrated_test_probs)
calibrated_auprc = average_precision_score(y_test, calibrated_test_probs)

print(f"Brier Score: {cal_metrics['brier_before']:.4f} -> {cal_metrics['brier_after']:.4f} "
      f"({cal_metrics['improvement_pct']:.1f}% improvement)")
print(f"\nCalibrated Test AUC:   {calibrated_auc:.4f}")
print(f"Calibrated Test AUPRC: {calibrated_auprc:.4f}")

In [ ]:
# ── Calibration Curve: Before vs After ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Calibration curve (reliability diagram)
ax = axes[0]
for name, probs, color, ls in [
    ('Before (raw)', raw_test_probs, '#e74c3c', '--'),
    ('After (calibrated)', calibrated_test_probs, '#2ecc71', '-'),
]:
    prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, marker='o', label=name, color=color, linewidth=2, linestyle=ls)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Perfect calibration')
ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Observed Frequency')
ax.set_title('Calibration Curve (Reliability Diagram)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Score distributions
ax = axes[1]
ax.hist(calibrated_test_probs[y_test == 0], bins=30, alpha=0.6, color='#2ecc71',
        label=f'Stable (n={int((y_test==0).sum())})', density=True)
ax.hist(calibrated_test_probs[y_test == 1], bins=30, alpha=0.6, color='#e74c3c',
        label=f'Delinquent (n={int((y_test==1).sum())})', density=True)
ax.set_xlabel('Calibrated PD Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by True Label', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Brier score comparison
ax = axes[2]
brier_raw = brier_score_loss(y_test, raw_test_probs)
brier_cal = brier_score_loss(y_test, calibrated_test_probs)
bars = ax.bar(['Raw Ensemble', 'Calibrated (Isotonic)'], [brier_raw, brier_cal],
              color=['#e74c3c', '#2ecc71'], edgecolor='black', linewidth=0.5)
ax.set_ylabel('Brier Score (lower is better)')
ax.set_title('Brier Score: Before vs After Calibration', fontweight='bold')
for bar, val in zip(bars, [brier_raw, brier_cal]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{val:.4f}',
            ha='center', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("After calibration, the model's predicted probabilities align with observed default rates.")
print("This is required for IFRS 9 Expected Credit Loss (ECL) calculations.")

---
## 8. Cost-Sensitive Threshold Optimization

**Business logic:** Missing a delinquent customer costs ~£10,000 (average loan loss).
An unnecessary RM call costs ~£5. An unnecessary SMS costs ~£0.50.

The optimal thresholds are driven by this 2000:1 cost asymmetry, not arbitrary cutoffs.

- **Critical tier** → RM call, restructuring offer (high intervention cost)
- **Watch tier** → Automated SMS/push nudge (low intervention cost)
- **Stable** → No action

In [ ]:
threshold_results = optimize_thresholds(y_test, calibrated_test_probs)
save_thresholds(threshold_results, os.path.join(MODEL_DIR, "thresholds.joblib"))

crit_t = threshold_results['critical_threshold']
watch_t = threshold_results['watch_threshold']

print(f"Critical threshold: {crit_t:.2f} (recall={threshold_results['critical_recall']:.1%})")
print(f"Watch threshold:    {watch_t:.2f} (recall={threshold_results['watch_recall']:.1%})")
print(f"\nCost assumptions:")
print(f"  Missed default (FN):       £{threshold_results['cost_assumptions']['missed_default']:,.0f}")
print(f"  Unnecessary RM call (FP):  £{threshold_results['cost_assumptions']['unnecessary_call']:.2f}")
print(f"  Unnecessary SMS (FP):      £{threshold_results['cost_assumptions']['unnecessary_sms']:.2f}")

In [ ]:
# ── Threshold Optimization Visualization ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Total cost vs threshold sweep
ax = axes[0]
thresholds_sweep = np.arange(0.01, 0.95, 0.01)
costs_critical = []
costs_watch = []
for t in thresholds_sweep:
    preds = (calibrated_test_probs >= t).astype(int)
    fn = ((preds == 0) & (y_test == 1)).sum()
    fp = ((preds == 1) & (y_test == 0)).sum()
    costs_critical.append(fn * 10000 + fp * 5)
    costs_watch.append(fn * 10000 + fp * 0.5)

ax.plot(thresholds_sweep, np.array(costs_critical)/1000, color='#e74c3c', linewidth=2, label='RM call (FP=£5)')
ax.plot(thresholds_sweep, np.array(costs_watch)/1000, color='#f39c12', linewidth=2, label='SMS (FP=£0.50)')
ax.axvline(x=crit_t, color='#e74c3c', linestyle='--', alpha=0.7, label=f'Critical={crit_t:.2f}')
ax.axvline(x=watch_t, color='#f39c12', linestyle='--', alpha=0.7, label=f'Watch={watch_t:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Total Cost (£k)')
ax.set_title('Cost-Sensitive Threshold Sweep', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. Confusion matrix at critical threshold
ax = axes[1]
crit_preds = (calibrated_test_probs >= crit_t).astype(int)
cm = confusion_matrix(y_test, crit_preds)
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title(f'Confusion Matrix (threshold={crit_t:.2f})', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Stable', 'Delinquent'])
ax.set_yticklabels(['Stable', 'Delinquent'])
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=14, fontweight='bold')

# 3. Risk tier distribution
ax = axes[2]
n_critical = int((calibrated_test_probs >= crit_t).sum())
n_watch = int(((calibrated_test_probs >= watch_t) & (calibrated_test_probs < crit_t)).sum())
n_stable = int((calibrated_test_probs < watch_t).sum())

tier_colors = ['#e74c3c', '#f39c12', '#2ecc71']
bars = ax.bar(['Critical', 'Watch', 'Stable'], [n_critical, n_watch, n_stable],
              color=tier_colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Number of Customers')
ax.set_title('Risk Tier Distribution (Test Set)', fontweight='bold')
for bar, val in zip(bars, [n_critical, n_watch, n_stable]):
    pct = val / len(calibrated_test_probs) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val}\n({pct:.1f}%)',
            ha='center', fontweight='bold', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Classification report at critical threshold
print("\nClassification Report (at critical threshold):")
print(classification_report(y_test, crit_preds, target_names=['Stable', 'Delinquent']))

---
## 9. SHAP Explainability

PRA SS1/23 and FCA Consumer Duty require model decisions to be explainable.
SHAP (SHapley Additive exPlanations) provides feature-level attributions for each prediction.

In [ ]:
import shap

# SHAP on XGBoost (tree-based, fast)
explainer = shap.TreeExplainer(xgb_model.get_booster())
shap_values = explainer.shap_values(X_test)

# Summary plot (global feature importance)
print("SHAP Summary Plot — Global Feature Importance")
print("Shows which features drive delinquency predictions across all test customers.")
shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=True, max_display=20)

In [ ]:
# SHAP bar plot (mean absolute SHAP values)
shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type='bar',
                  show=True, max_display=15)

In [ ]:
# ── Single Customer Explanation (highest risk) ──
highest_risk_idx = np.argmax(calibrated_test_probs)
print(f"Highest-risk customer in test set:")
print(f"  Calibrated PD: {calibrated_test_probs[highest_risk_idx]:.3f}")
print(f"  True label: {'DELINQUENT' if y_test[highest_risk_idx] == 1 else 'STABLE'}")
print(f"  Risk tier: {ensemble.score_to_risk_tier(calibrated_test_probs[highest_risk_idx])}")
print()

# Waterfall plot for this customer
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[highest_risk_idx],
        base_values=explainer.expected_value,
        data=X_test[highest_risk_idx],
        feature_names=feature_names
    ),
    max_display=12
)

---
## 10. Phase 2 Models — Survival Analysis, Conformal Prediction, Uplift

In [ ]:
# ── Survival Model (CoxPH — Time-to-Event) ──
try:
    from ml.survival_model import SurvivalModel
    surv_model = SurvivalModel()
    surv_features = pd.DataFrame(X_train, columns=feature_names)
    surv_features["risk_score"] = xgb_model.predict_proba(X_train)
    surv_features["event"] = y_train
    surv_features["duration"] = np.where(
        y_train == 1,
        np.random.uniform(5, 60, len(y_train)),
        np.random.uniform(60, 180, len(y_train)),
    )
    survival_metrics = surv_model.fit(surv_features, duration_col="duration", event_col="event")
    surv_model.save(os.path.join(MODEL_DIR, "survival_model.joblib"))
    print(f"Survival Model — Concordance Index: {survival_metrics.get('concordance', 'N/A'):.4f}")
    print("  Provides time-to-event (TTE) estimates: 'this customer will likely default in ~23 days'")
except Exception as e:
    print(f"Survival model error (non-fatal): {e}")

In [ ]:
# ── Conformal Predictor (Uncertainty Intervals) ──
try:
    from ml.conformal import ConformalPredictor
    conformal = ConformalPredictor(confidence=0.90)
    conformal_metrics = conformal.calibrate(y_cal, calibrator.calibrate_batch(cal_ensemble_probs))
    conformal.save(os.path.join(MODEL_DIR, "conformal_predictor.joblib"))
    print(f"Conformal Predictor — 90% Confidence Intervals")
    print(f"  Empirical coverage: {conformal_metrics.get('empirical_coverage', 'N/A')}")
    print("  Provides: 'PD = 0.42 [0.31, 0.55] at 90% confidence'")
except Exception as e:
    print(f"Conformal prediction error (non-fatal): {e}")

In [ ]:
# ── Uplift Model (Treatment Effect) ──
try:
    from ml.uplift_model import UpliftModel
    uplift = UpliftModel()
    treatment_mask = np.random.binomial(1, 0.5, len(X_train)).astype(bool)
    uplift_metrics = uplift.fit(X_train, y_train, treatment_mask, feature_names=feature_names)
    uplift.save(os.path.join(MODEL_DIR, "uplift_model.joblib"))
    print(f"Uplift Model (T-Learner)")
    print(f"  Mean uplift: {uplift_metrics.get('mean_uplift', 'N/A'):.4f}")
    print("  Answers: 'will intervening with THIS customer actually reduce their default probability?'")
except Exception as e:
    print(f"Uplift model error (non-fatal): {e}")

---
## 11. Fairness Audit (FCA Consumer Duty Compliance)

FCA Consumer Duty requires demonstrating that models do not produce
disparate outcomes across protected characteristics (age, gender, region, income).

In [ ]:
try:
    from ml.fairness import run_bias_audit
    from sqlalchemy import create_engine

    engine = create_engine(PostgresConfig.get_url())
    demo_df = pd.read_sql(
        "SELECT customer_id, age, gender, region, income_bracket FROM customers", engine
    )
    test_cids = customer_ids[idx_test]
    demo_test = demo_df[demo_df["customer_id"].isin(test_cids)].reset_index(drop=True)

    if len(demo_test) >= len(X_test):
        demo_test = demo_test.iloc[:len(X_test)]
        fairness_results = run_bias_audit(xgb_model, X_test, y_test, demo_test)
        print(f"Fairness Verdict: {fairness_results.get('verdict', 'N/A')}")
        print(f"Frameworks used:  {fairness_results.get('frameworks_used', [])}")
        if 'metrics' in fairness_results:
            for attr, metrics in fairness_results.get('metrics', {}).items():
                print(f"  {attr}: {metrics}")
    else:
        print("Insufficient demographic data for fairness audit")
except Exception as e:
    print(f"Fairness audit error (non-fatal): {e}")

---
## 12. End-to-End Scoring Demo — Single Customer

In [ ]:
# Pick a delinquent and a stable customer from the test set
delinquent_indices = np.where(y_test == 1)[0]
stable_indices = np.where(y_test == 0)[0]

demo_indices = []
demo_labels = []
if len(delinquent_indices) > 0:
    # Pick the highest-risk delinquent
    del_idx = delinquent_indices[np.argmax(calibrated_test_probs[delinquent_indices])]
    demo_indices.append(del_idx)
    demo_labels.append('DELINQUENT')
if len(stable_indices) > 0:
    # Pick the lowest-risk stable
    stb_idx = stable_indices[np.argmin(calibrated_test_probs[stable_indices])]
    demo_indices.append(stb_idx)
    demo_labels.append('STABLE')

for idx, true_label in zip(demo_indices, demo_labels):
    xgb_p = float(xgb_test_probs[idx])
    lgb_p = float(lgb_test_probs[idx])
    tft_p = float(tft_test_probs[idx]) if tft_test_probs is not None else None
    raw_ensemble = float(ensemble.combine(xgb_prob=xgb_p, lgb_prob=lgb_p, tft_prob=tft_p))
    calibrated_pd = float(calibrated_test_probs[idx])
    tier = ensemble.score_to_risk_tier(calibrated_pd)
    credit_eq = ensemble.score_to_credit_score(calibrated_pd)

    print(f"{'='*60}")
    print(f"  Customer: test_sample_{idx} | True Label: {true_label}")
    print(f"{'='*60}")
    print(f"  XGBoost score:       {xgb_p:.4f}")
    print(f"  LightGBM score:      {lgb_p:.4f}")
    if tft_p is not None:
        print(f"  TFT score:           {tft_p:.4f}")
    print(f"  Raw ensemble:        {raw_ensemble:.4f}")
    print(f"  Calibrated PD:       {calibrated_pd:.4f}")
    print(f"  Risk Tier:           {tier.upper()}")
    print(f"  Credit Score (eq):   {credit_eq}")
    print(f"  Critical threshold:  {crit_t:.2f}")
    print(f"  Watch threshold:     {watch_t:.2f}")

    # Top SHAP drivers for this customer
    sv = shap_values[idx]
    top_indices = np.argsort(np.abs(sv))[::-1][:5]
    print(f"  Top SHAP drivers:")
    for rank, fi in enumerate(top_indices):
        direction = 'increases' if sv[fi] > 0 else 'decreases'
        print(f"    {rank+1}. {feature_names[fi]}: {sv[fi]:+.4f} ({direction} risk)")
    print()

---
## 13. Summary — Model Performance Card

In [ ]:
print("=" * 70)
print("  BARCLAYS PRE-DELINQUENCY ENGINE — MODEL PERFORMANCE CARD")
print("=" * 70)
print()
print(f"  Dataset")
print(f"  {'─'*40}")
print(f"  Total customers:       {len(y_tab):,}")
print(f"  Delinquency rate:      {y_tab.mean()*100:.1f}%")
print(f"  Features:              {len(feature_names)}")
print(f"  Labels source:         payment_events (outcome-based)")
print(f"  Split:                 60/20/20 (train/calibration/test)")
print()
print(f"  Individual Models (Test Set)")
print(f"  {'─'*40}")
print(f"  XGBoost AUC:           {roc_auc_score(y_test, xgb_test_probs):.4f}")
print(f"  XGBoost AUPRC:         {xgb_auprc:.4f}")
print(f"  LightGBM AUC:          {roc_auc_score(y_test, lgb_test_probs):.4f}")
print(f"  LightGBM AUPRC:        {lgb_auprc:.4f}")
if tft_test_probs is not None:
    tft_auc = roc_auc_score(y_test, tft_test_probs)
    tft_ap = average_precision_score(y_test, tft_test_probs)
    print(f"  TFT AUC:              {tft_auc:.4f}")
    print(f"  TFT AUPRC:            {tft_ap:.4f}")
print()
print(f"  Ensemble (Test Set)")
print(f"  {'─'*40}")
print(f"  Ensemble AUC:          {ensemble_auc:.4f}")
print(f"  Ensemble AUPRC:        {ensemble_auprc:.4f}")
print(f"  Calibrated AUC:        {calibrated_auc:.4f}")
print(f"  Calibrated AUPRC:      {calibrated_auprc:.4f}")
print()
print(f"  Calibration")
print(f"  {'─'*40}")
print(f"  Brier (before):        {cal_metrics['brier_before']:.4f}")
print(f"  Brier (after):         {cal_metrics['brier_after']:.4f}")
print(f"  Improvement:           {cal_metrics['improvement_pct']:.1f}%")
print()
print(f"  Cost-Optimized Thresholds")
print(f"  {'─'*40}")
print(f"  Critical threshold:    {crit_t:.2f} (recall={threshold_results['critical_recall']:.1%})")
print(f"  Watch threshold:       {watch_t:.2f} (recall={threshold_results['watch_recall']:.1%})")
print(f"  Cost ratio:            FN=£10,000 vs FP=£5 (critical) / £0.50 (watch)")
print()
if meta_metrics:
    print(f"  Meta-Learner")
    print(f"  {'─'*40}")
    print(f"  CV AUC:               {meta_metrics.get('cv_auc_mean', 0):.4f} +/- {meta_metrics.get('cv_auc_std', 0):.4f}")
    print(f"  Training method:       5-fold OOF (no leakage)")
    print()
print(f"  Regulatory Compliance")
print(f"  {'─'*40}")
print(f"  IFRS 9 PD:             Calibrated (isotonic regression)")
print(f"  PRA SS1/23:            SHAP explainability")
print(f"  FCA Consumer Duty:     Fairness audit (Fairlearn + AIF360)")
print(f"  Basel III/IV:          Cost-sensitive thresholds")
print("=" * 70)

In [ ]:
# ── Final: Save all models ──
print("Models saved to:", MODEL_DIR)
for f in os.listdir(MODEL_DIR):
    fpath = os.path.join(MODEL_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f:40s} {size_kb:>8.1f} KB")